# Paper §4 main panel — cross-model × cross-dataset reproduction

**Goal.** Reproduce, from raw model predictions, the 6-model × 5-dataset
phenomenon panel that underpins paper §4 (Anchoring effect: graded pull
and literal copy) and Appendix D.1 / D.2.

**Data location.** `outputs/paper/cross_model_cross_dataset/` (this
notebook reads it top-to-bottom, no other dependencies).

**Spec source-of-truth.** `docs/paper/emnlp_outline_ko.md` §3.4 (panel),
§B (filters), §C (anchors).

Every hyperparameter, prompt, filter, and anchor rule used to produce the
predictions is written *explicitly* in the next four markdown cells, so
this notebook is self-contained — no need to chase configs.


## 1 · Hyperparameters (identical across all 6 models, all 5 datasets, all 4 conditions)

| Knob | Value |
|---|---|
| Random seed (filter / anchor RNG / neutral RNG) | `42` |
| Sampling temperature | `0.0` (greedy) |
| Sampling top_p | `1.0` |
| `max_new_tokens` | `16` (TallyQA: `8`) |
| Decoding | greedy, single shot, no self-consistency |
| Image preprocessing | each model's native HF processor — no resizing/cropping override |
| `attn_implementation` | model default (eager for some, SDPA for others — see `HFAttentionRunner`) |


## 2 · Prompt template (identical across all cells)

**System prompt:**

```
You are a visual question answering system.
Return valid JSON only in the form {"result": <number>}.
Use a numeric JSON value for <number>, not a string.
Do not output any other keys, words, explanation, or markdown.
If uncertain, still output the single most likely number in that JSON format.
```

**User template:**

```
Answer the question using the provided image(s).
Return JSON only in the form {"result": <number>}.
Question: {question}
```

Number of images attached:
- `b` (target_only): 1 image
- `a` / `m` / `d`: 2 images (target + distractor)


## 3 · Dataset filters (n_b = eligible base samples)

| Dataset | Split | `answer_type` filter | GT range | n_eligible |
|---|---|---|---:|---:|
| TallyQA        | `test`     | `number`  | [0, 8]     | 38,245 |
| ChartQA        | `test`     | `text`¹   | [0, 1000]  | 705 |
| MathVista      | `testmini` | `integer` | [0, 1000]  | 385 |
| PlotQA         | `test`     | —         | [0, 10000] | 5,000² |
| InfographicVQA | `val`      | —         | [0, 10000] | 1,147 |

¹ ChartQA rows carry `answer_type="text"`; the integer-GT gate inside
`load_number_vqa_samples` keeps only the numeric-answerable subset.
² PlotQA JSONL snapshot is a pre-stratified 5,000-sample subset (seed=42,
5 GT bins × 1,000 each — built once at fetch time, see
`scripts/build_plotqa_test_subset.py`).

All datasets also satisfy `require_single_numeric_gt=True` — only samples
whose all GT candidates are the same integer survive the filter.


## 4 · Anchor selection (per outline §C)

**Inventory: 128 pre-rendered digit images** at `inputs/irrelevant_number/{value}.png` —
- 0–10 step 1 (11 values) · 15–100 step 5 (18 values) · 200–10,000 step 100 (99 values).

**Per-question stratification (single S1 stratum, close-distance):**

| Dataset | Scheme | Effective constraint on |a − gt| |
|---|---|---|
| TallyQA        | `absolute`    | ≤ 5 (fixed) |
| ChartQA        | `relative_s1` | ≤ max(1, ⌊0.10·gt⌋) |
| MathVista      | `relative_s1` | ≤ max(1, ⌊0.10·gt⌋) |
| PlotQA         | `relative_s1` | ≤ max(1, ⌊0.10·gt⌋) |
| InfographicVQA | `relative_s1` | ≤ max(1, ⌊0.10·gt⌋) |

For each question, `random.Random(seed=42)` picks one anchor uniformly from
the inventory candidates satisfying the distance constraint. If the inventory
has no candidate inside the constraint (e.g., GT=12 with relative_s1 needs
|a−12|≤1 but inventory has step=1 only up to 10 then jumps to 15), the
question contributes only `b` and `d` arms — no `a` / `m`.

Implementation: `src/vlm_anchor/data.py` → `compute_strata()` + `sample_stratified_anchors()`.


## 5 · Metrics (C-form, current canonical) — restricted to base-wrong subset

Outline §3.3 defines two subsets:
- **base-correct** = { i | pb_i == gt_i }
- **base-wrong**   = { i | pb_i != gt_i }

The headline panel numbers (outline §D.1 / §D.2) are computed on the
**base-wrong** subset only — anchoring is meaningful exactly when the
model isn't already right at baseline. This convention is set in
`scripts/build_e5e_e7_5dataset_summary.py::_wrong_base_s1()`.

Within the base-wrong subset, for each sample i with base prediction pb,
condition-c prediction p^c, anchor digit z, ground truth gt:

$$
\text{Adopt}_c     = \frac{ \#\{i : p^c_i = z_i \text{ and } pb_i \neq z_i\} }{ \#\{i : pb_i \neq z_i\} }
$$

$$
\text{DF}_c = \frac{ \#\{i : (p^c_i - pb_i)(z_i - pb_i) > 0 \text{ and } p^c_i \neq pb_i\} }{ \#\{i : \text{numeric pair and anchor present}\} }
$$

$$
\text{EM}_c        = \frac{ \#\{i : p^c_i = gt_i\} }{ \#\{i : \text{numeric}\} }
$$

> **Note.** Outline §3.3 finalizes DF to the epsilon-threshold form
> `P[(p^c−pb)(z−pb) > eps | |z−pb| > eps]`; the current canonical numbers
> use the C-form above. Both are sign-based and gt-free.


## 6 · Setup

In [1]:
from __future__ import annotations
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

# Allow running this notebook from either repo root or notebooks/.
THIS = Path.cwd()
REPO = THIS if (THIS / "outputs" / "paper" / "cross_model_cross_dataset").exists() else THIS.parent
PAPER_DIR = REPO / "outputs" / "paper" / "cross_model_cross_dataset"
PRED_ROOT = PAPER_DIR / "predictions"
CANON_PER_CELL = REPO / "docs" / "insights" / "_data" / "main_panel_5dataset_per_cell.csv"

assert PAPER_DIR.exists(), f"missing: {PAPER_DIR}"

MODELS = [
    "llava-onevision-qwen2-7b-ov",   # Main
    "llava-next-interleaved-7b",
    "qwen2.5-vl-7b-instruct",
    "qwen2.5-vl-32b-instruct",
    "gemma3-4b-it",
    "gemma3-27b-it",
]
DATASETS = ["tallyqa", "chartqa", "mathvista", "plotqa", "infographicvqa"]
DISPLAY = {
    "tallyqa": "TallyQA",
    "chartqa": "ChartQA",
    "mathvista": "MathVista",
    "plotqa": "PlotQA",
    "infographicvqa": "InfographicVQA",
}

# Outline-spec eligible n_b per dataset
N_B_EXPECTED = {"tallyqa": 38245, "chartqa": 705, "mathvista": 385, "plotqa": 5000, "infographicvqa": 1147}

# Outline-spec anchor scheme per dataset
ANCHOR_SCHEME = {
    "tallyqa":        ("absolute",    (0, 5)),
    "chartqa":        ("relative_s1", None),
    "mathvista":      ("relative_s1", None),
    "plotqa":         ("relative_s1", None),
    "infographicvqa": ("relative_s1", None),
}

print(f"REPO  = {REPO}")
print(f"PAPER = {PAPER_DIR}")


REPO  = /mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing
PAPER = /mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing/outputs/paper/cross_model_cross_dataset


## 7 · How `predictions.csv` is generated — full pipeline

The 30 `predictions.csv` snapshots loaded later in this notebook were
produced by `scripts/run_experiment.py`. Below we walk through the
**same five-step pipeline** using the `vlm_anchor` library, so a reviewer
can see exactly how a `(model × dataset)` cell is constructed end-to-end:

1. **Load eligible samples** — apply outline §B dataset filters via
   `vlm_anchor.data.load_number_vqa_samples`.
2. **Assign stratified anchors** — outline §C scheme via
   `vlm_anchor.data.assign_stratified_anchors` (`seed=42`).
3. **Build 4 conditions per sample** (`b` / `a-S1` / `m-S1` / `d`) via
   `vlm_anchor.data.build_conditions`.
4. **Run model inference** — greedy decode via
   `vlm_anchor.models.HFAttentionRunner.generate_number` (requires GPU;
   gated by `RUN_INFERENCE` toggle below).
5. **Evaluate per row** — populate canonical per-row flags via
   `vlm_anchor.metrics.evaluate_sample`; these are the same flags
   aggregated in §11 / §12 / §13 below.

The demo runs on **TallyQA × Qwen2.5-VL-7B × 3 samples** to keep wall-time
low. Full-scale reproduction (all 30 cells) is one
`scripts/run_experiment.py` invocation per dataset (see §7.7 below).


### 7.1 · Demo configuration

In [2]:
DEMO_DATASET    = "tallyqa"
DEMO_MODEL_NAME = "qwen2.5-vl-7b-instruct"
DEMO_HF_MODEL   = "Qwen/Qwen2.5-VL-7B-Instruct"
DEMO_N_SAMPLES  = 3
RUN_INFERENCE   = False     # set True to actually load model + run generate (GPU required)

# Dataset-specific args matching configs/experiment_e5e_tallyqa_full.yaml
DEMO_DATASET_LOCAL = REPO / "inputs" / "tallyqa_test"
DEMO_DATASET_KWARGS = dict(
    answer_type_filter=["number"],
    answer_range=8,
    require_single_numeric_gt=True,
)
DEMO_ANCHOR_KWARGS  = dict(scheme="absolute", strata=[(0, 5)])
DEMO_MAX_NEW_TOKENS = 8


### 7.2 · Step 1 — load eligible samples

In [3]:
import sys; sys.path.insert(0, str(REPO / "src"))
from vlm_anchor.data import (
    load_number_vqa_samples, assign_stratified_anchors, build_conditions,
)

samples = load_number_vqa_samples(
    dataset_path=DEMO_DATASET_LOCAL,
    max_samples=DEMO_N_SAMPLES,
    **DEMO_DATASET_KWARGS,
)
print(f"Loaded {len(samples)} eligible samples from {DEMO_DATASET_LOCAL}")
samples[0]


Loaded 3 eligible samples from /mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing/inputs/tallyqa_test


{'question_id': 0,
 'image_id': 0,
 'question': 'How many people are there?',
 'image': PosixPath('/mnt/ddn/prod-runs/thyun.park/src/vlm_anchroing/inputs/tallyqa_test/images/000000000000.png'),
 'ground_truth': '2',
 'answers': ['2'],
 'question_type': 'how many'}

### 7.3 · Step 2 — assign stratified anchors (seed=42)

In [4]:
samples_with_anchors = assign_stratified_anchors(
    samples,
    irrelevant_number_dir=REPO / "inputs" / "irrelevant_number",
    irrelevant_number_masked_dir=REPO / "inputs" / "irrelevant_number_masked",
    irrelevant_neutral_dir=REPO / "inputs" / "irrelevant_neutral",
    seed=42,
    **DEMO_ANCHOR_KWARGS,
)
# Verify anchor distance respects the dataset's scheme
for s in samples_with_anchors:
    gt = int(s["ground_truth"])
    for entry in s["anchor_strata"]:
        z = entry["anchor_value"]
        print(f"  qid={s['question_id']} gt={gt:>2}  stratum={entry['stratum_id']}  z={z}  |z-gt|={abs(z-gt) if z is not None else 'NA'}")


  qid=0 gt= 2  stratum=S1  z=1  |z-gt|=1
  qid=1 gt= 2  stratum=S1  z=4  |z-gt|=2
  qid=2 gt= 2  stratum=S1  z=3  |z-gt|=1


### 7.4 · Step 3 — build 4 conditions per sample

In [5]:
all_conds = []
for s in samples_with_anchors:
    for c in build_conditions(s):
        all_conds.append(c)
print(f"{len(all_conds)} (sample, condition) rows from {len(samples_with_anchors)} samples")
pd.DataFrame([
    {"sid": c["sample_instance_id"], "condition": c["condition"],
     "n_images": len(c["input_images"]), "anchor": c.get("anchor_value_for_metrics")}
    for c in all_conds
])


12 (sample, condition) rows from 3 samples


,sid,condition,n_images,anchor
0,0_0_stratified,target_only,1,NaN
1,0_0_stratified,target_plus_irrelevant_number_S1,2,1
2,0_0_stratified,target_plus_irrelevant_number_masked_S1,2,1
3,0_0_stratified,target_plus_irrelevant_neutral,2,NaN
4,1_0_stratified,target_only,1,NaN
5,1_0_stratified,target_plus_irrelevant_number_S1,2,4
6,1_0_stratified,target_plus_irrelevant_number_masked_S1,2,4
7,1_0_stratified,target_plus_irrelevant_neutral,2,NaN
8,2_0_stratified,target_only,1,NaN
9,2_0_stratified,target_plus_irrelevant_number_S1,2,3


### 7.5 · Step 4 — run model inference (greedy decode)

This cell loads the model and generates a prediction per `(sample, condition)`
row using `HFAttentionRunner.generate_number` — the same code path that
`scripts/run_experiment.py` uses. Skipped when `RUN_INFERENCE=False`.


In [6]:
if RUN_INFERENCE:
    from vlm_anchor.models import HFAttentionRunner, InferenceConfig
    cfg = InferenceConfig(
        system_prompt=manifest["hyperparameters"]["prompt_system"],
        user_template=manifest["hyperparameters"]["prompt_user_template"],
        temperature=0.0,
        top_p=1.0,
        max_new_tokens=DEMO_MAX_NEW_TOKENS,
    )
    runner = HFAttentionRunner(DEMO_HF_MODEL, inference_config=cfg)

    inference_rows = []
    for c in all_conds:
        out = runner.generate_number(
            question=c["question"],
            images=c["input_images"],
            max_new_tokens=DEMO_MAX_NEW_TOKENS,
        )
        inference_rows.append({**c, **out})
    DEMO_INFER = pd.DataFrame([
        {"sid": r["sample_instance_id"], "condition": r["condition"],
         "raw": r["raw_prediction"], "prediction": r["prediction"]}
        for r in inference_rows
    ])
    print(f"Generated {len(inference_rows)} predictions")
else:
    inference_rows = None
    DEMO_INFER = None
    print("RUN_INFERENCE=False — skipping demo inference.")
    print("To reproduce live: set RUN_INFERENCE=True (GPU required) and re-run from §7.1.")
DEMO_INFER


RUN_INFERENCE=False — skipping demo inference.
To reproduce live: set RUN_INFERENCE=True (GPU required) and re-run from §7.1.


### 7.6 · Step 5 — evaluate per-row flags

`evaluate_sample` computes the canonical per-row indicators
(`exact_match`, `anchor_adopted`, `anchor_direction_followed_moved`,
`pred_b_equal_anchor`, `numeric_distance_to_anchor`) — these are what
the predictions.csv files carry, and what the analysis sections aggregate.


In [7]:
if RUN_INFERENCE:
    from vlm_anchor.metrics import evaluate_sample
    # pred_b (base prediction) per sid — needed by evaluate_sample
    base_by_sid = {r["sample_instance_id"]: r["prediction"]
                   for r in inference_rows if r["condition"] == "target_only"}
    eval_rows = []
    for r in inference_rows:
        ev = evaluate_sample(
            prediction=r["prediction"],
            gt_answer=r["ground_truth"],
            all_answers=r["answers"],
            anchor_value=r.get("anchor_value_for_metrics"),
            base_prediction=base_by_sid.get(r["sample_instance_id"]),
        )
        eval_rows.append({
            "sid": r["sample_instance_id"],
            "condition": r["condition"],
            "gt": r["ground_truth"],
            "z": r.get("anchor_value_for_metrics"),
            "prediction": r["prediction"],
            "exact_match": ev.exact_match,
            "anchor_adopted": ev.anchor_adopted,
            "anchor_direction_followed_moved": ev.anchor_direction_followed_moved,
            "pred_b_equal_anchor": ev.pred_b_equal_anchor,
            "numeric_distance_to_anchor": ev.numeric_distance_to_anchor,
        })
    DEMO_EVAL = pd.DataFrame(eval_rows)
else:
    DEMO_EVAL = None
    print("(skipped — needs DEMO_INFER from §7.5)")
DEMO_EVAL


(skipped — needs DEMO_INFER from §7.5)


### 7.7 · Full reproduction — all 30 cells (6 models × 5 datasets)

The notebook above demos one cell × 3 samples. Reproducing the full panel
is one shell command per dataset:

```bash
uv run python scripts/run_experiment.py --config configs/experiment_e5e_tallyqa_full.yaml
uv run python scripts/run_experiment.py --config configs/experiment_e5e_chartqa_full.yaml
uv run python scripts/run_experiment.py --config configs/experiment_e5e_mathvista_full.yaml
uv run python scripts/run_experiment.py --config configs/experiment_e7_plotqa_full.yaml
uv run python scripts/run_experiment.py --config configs/experiment_e7_infographicvqa_full.yaml
```

Each config pins the 6 models, seed, sampling params, dataset filters, and
anchor scheme described in §1–§5. After all five complete, regenerate the
aggregator CSV with:

```bash
uv run python scripts/build_e5e_e7_5dataset_summary.py
```

…which produces `docs/insights/_data/main_panel_5dataset_per_cell.csv`
(the canonical aggregator that the §15 cross-check below verifies against).

Approximate compute: ~24–48 GPU-hours total on A100 80 GB (dominated by
TallyQA n=38,245 across 6 models).


## 8 · Manifest — lineage of the existing 30 cells

In [8]:
manifest = json.loads((PAPER_DIR / "manifest.json").read_text())
print(f"Generated at: {manifest['generated_at']}")
print(f"Spec ref:     {manifest['spec_reference']}\n")
print(f"#cells = {len(manifest['cells'])}  (expected 30)")
pd.DataFrame(manifest["cells"])[
    ["dataset", "model", "source_exp_dir", "source_run", "n_b_eligible", "predictions_size_bytes"]
].head(10)


Generated at: 2026-05-16T17:36:58Z
Spec ref:     docs/paper/emnlp_outline_ko.md §3.4 (panel definition) + §B (dataset filters) + §C (anchor selection)

#cells = 30  (expected 30)


,dataset,model,source_exp_dir,source_run,n_b_eligible,predictions_size_bytes
0,TallyQA,gemma3-27b-it,experiment_e5e_tallyqa_full,20260428-003945,38245,162448853
1,TallyQA,gemma3-4b-it,experiment_e5e_tallyqa_full,20260503-165102,38245,180417744
2,TallyQA,llava-next-interleaved-7b,experiment_e5e_tallyqa_full,20260427-171240,38245,100421561
3,TallyQA,llava-onevision-qwen2-7b-ov,experiment_e5e_tallyqa_full,20260502-083926,38245,110990693
4,TallyQA,qwen2.5-vl-32b-instruct,experiment_e5e_tallyqa_full,20260503-113654,38245,190531732
5,TallyQA,qwen2.5-vl-7b-instruct,experiment_e5e_tallyqa_full,20260427-235812,38245,176864744
6,ChartQA,gemma3-27b-it,experiment_e5e_chartqa_full,20260428-001359,705,3059643
7,ChartQA,gemma3-4b-it,experiment_e5e_chartqa_full,20260503-200044,705,3467071
8,ChartQA,llava-next-interleaved-7b,experiment_e5e_chartqa_full,20260427-171240,705,2087188
9,ChartQA,llava-onevision-qwen2-7b-ov,experiment_e5e_chartqa_full,20260502-211028,705,2220375


## 9 · Load all 30 existing predictions.csv into one long-format DataFrame

In [9]:
LOAD_COLS = [
    # Identifiers + raw row
    "sample_instance_id", "question_id", "image_id",
    "condition", "irrelevant_type", "anchor_stratum_id",
    "ground_truth", "anchor_value", "prediction",
    # Per-row flags computed by src/vlm_anchor/metrics.py::evaluate_sample
    # (see also outline §3.3). Aggregating these matches the canonical
    # build_e5e_e7_5dataset_summary.py output exactly.
    "exact_match", "anchor_adopted",
    "anchor_direction_followed_moved", "pred_b_equal_anchor",
    "numeric_distance_to_anchor",
]

frames = []
for ds in DATASETS:
    for model in MODELS:
        p = PRED_ROOT / ds / model / "predictions.csv"
        df = pd.read_csv(p, usecols=LOAD_COLS, low_memory=False)
        df["dataset"] = DISPLAY[ds]
        df["model"]   = model
        frames.append(df)

ALL = pd.concat(frames, ignore_index=True)
print(f"Total rows = {len(ALL):,}  (≈ 4 conditions × n_eligible × 6 models per dataset)")
print(f"Datasets   = {ALL['dataset'].unique().tolist()}")
print(f"Models     = {ALL['model'].nunique()}")
print(f"Conditions = {sorted(ALL['condition'].unique())}")
ALL.head(3)


Total rows = 1,086,084  (≈ 4 conditions × n_eligible × 6 models per dataset)
Datasets   = ['TallyQA', 'ChartQA', 'MathVista', 'PlotQA', 'InfographicVQA']
Models     = 6
Conditions = ['target_only', 'target_plus_irrelevant_neutral', 'target_plus_irrelevant_number_S1', 'target_plus_irrelevant_number_masked_S1']


,sample_instance_id,question_id,image_id,condition,irrelevant_type,ground_truth,prediction,anchor_value,anchor_stratum_id,exact_match,anchor_adopted,anchor_direction_followed_moved,pred_b_equal_anchor,numeric_distance_to_anchor,dataset,model
0,0_0_stratified,0,0,target_only,none,2,2,NaN,NaN,1,0,0,0,NaN,TallyQA,llava-onevision-qwen2-7b-ov
1,100_1_stratified,100,1,target_only,none,2,2,NaN,NaN,1,0,0,0,NaN,TallyQA,llava-onevision-qwen2-7b-ov
2,200_2_stratified,200,2,target_only,none,4,3,NaN,NaN,0,0,0,0,NaN,TallyQA,llava-onevision-qwen2-7b-ov


## 10 · Sanity checks (assert outline spec compliance)

Three checks per cell:

1. **n_b matches outline §B.** Unique `sample_instance_id`s under the `target_only` condition equals the spec's `n_eligible`.
2. **Per-condition row counts.** `target_only` and `target_plus_irrelevant_neutral` should both equal n_b. The two anchor conditions (`_S1` and `_masked_S1`) equal n_b *minus* samples whose inventory had no anchor inside the dataset's distance bound — they always match each other (paired).
3. **Anchor distance.** Every row in `target_plus_irrelevant_number_S1` satisfies the dataset's anchor scheme (`|a−gt|≤5` for TallyQA, `≤max(1, ⌊0.10·gt⌋)` for the four relative_s1 datasets).


In [10]:
def _relative_s1_bound(gt: int) -> int:
    return max(1, int(0.10 * abs(gt)))

rows = []
for ds in DATASETS:
    for model in MODELS:
        sub = ALL[(ALL["dataset"] == DISPLAY[ds]) & (ALL["model"] == model)]
        n_b_obs = sub.loc[sub["condition"] == "target_only", "sample_instance_id"].nunique()
        n_neu   = sub.loc[sub["condition"] == "target_plus_irrelevant_neutral", "sample_instance_id"].nunique()
        n_a     = sub.loc[sub["condition"] == "target_plus_irrelevant_number_S1", "sample_instance_id"].nunique()
        n_m     = sub.loc[sub["condition"] == "target_plus_irrelevant_number_masked_S1", "sample_instance_id"].nunique()

        # Anchor distance check on the S1 anchor rows
        s1 = sub[sub["condition"] == "target_plus_irrelevant_number_S1"].copy()
        s1["gt_i"] = pd.to_numeric(s1["ground_truth"], errors="coerce").astype("Int64")
        s1["z_i"]  = pd.to_numeric(s1["anchor_value"], errors="coerce").astype("Int64")
        ok = s1.dropna(subset=["gt_i", "z_i"]).copy()
        ok["dist"] = (ok["z_i"] - ok["gt_i"]).abs()

        scheme, abs_bounds = ANCHOR_SCHEME[ds]
        if scheme == "absolute":
            lo, hi = abs_bounds
            bound_ok = ((ok["dist"] >= lo) & (ok["dist"] <= hi)).all()
            scheme_desc = f"|a-gt| in [{lo}, {hi}]"
        else:
            bounds = ok["gt_i"].map(_relative_s1_bound).astype("Int64")
            bound_ok = (ok["dist"] <= bounds).all()
            scheme_desc = "|a-gt| <= max(1, int(0.10*gt))"

        rows.append({
            "dataset": DISPLAY[ds],
            "model": model,
            "n_b_obs": n_b_obs,
            "n_b_expected": N_B_EXPECTED[ds],
            "n_b_match": n_b_obs == N_B_EXPECTED[ds],
            "n_neutral": n_neu,
            "n_anchor_S1": n_a,
            "n_masked_S1": n_m,
            "a_m_paired": n_a == n_m,
            "scheme": scheme_desc,
            "anchor_dist_ok": bool(bound_ok),
        })

CHECK = pd.DataFrame(rows)
assert CHECK["n_b_match"].all(),       "Some cells have wrong n_b — check filters!"
assert CHECK["a_m_paired"].all(),      "a/m pairing broken — check anchor assignment!"
assert CHECK["anchor_dist_ok"].all(),  "Some anchor distances violate the dataset scheme!"
print("All sanity checks passed.")
CHECK


All sanity checks passed.


,dataset,model,n_b_obs,n_b_expected,n_b_match,n_neutral,n_anchor_S1,n_masked_S1,a_m_paired,scheme,anchor_dist_ok
0,TallyQA,llava-onevision-qwen2-7b-ov,38245,38245,True,38245,38245,38245,True,"|a-gt| in [0, 5]",True
1,TallyQA,llava-next-interleaved-7b,38245,38245,True,38245,38245,38245,True,"|a-gt| in [0, 5]",True
2,TallyQA,qwen2.5-vl-7b-instruct,38245,38245,True,38245,38245,38245,True,"|a-gt| in [0, 5]",True
3,TallyQA,qwen2.5-vl-32b-instruct,38245,38245,True,38245,38245,38245,True,"|a-gt| in [0, 5]",True
4,TallyQA,gemma3-4b-it,38245,38245,True,38245,38245,38245,True,"|a-gt| in [0, 5]",True
5,TallyQA,gemma3-27b-it,38245,38245,True,38245,38245,38245,True,"|a-gt| in [0, 5]",True
6,ChartQA,llava-onevision-qwen2-7b-ov,705,705,True,705,632,632,True,"|a-gt| <= max(1, int(0.10*gt))",True
7,ChartQA,llava-next-interleaved-7b,705,705,True,705,632,632,True,"|a-gt| <= max(1, int(0.10*gt))",True
8,ChartQA,qwen2.5-vl-7b-instruct,705,705,True,705,632,632,True,"|a-gt| <= max(1, int(0.10*gt))",True
9,ChartQA,qwen2.5-vl-32b-instruct,705,705,True,705,632,632,True,"|a-gt| <= max(1, int(0.10*gt))",True


## 11 · Compute per-cell metrics (broad + base-wrong)

Per outline §5 formulas (C-form). `predictions.csv` already carries the
per-row indicator flags computed by `src/vlm_anchor/metrics.py::evaluate_sample`:

| Per-row flag | Meaning |
|---|---|
| `exact_match`                       | `1` iff `p^c == gt` (int parse) |
| `anchor_adopted`                    | `1` iff `(pa == z) AND (pb != z)` AND `pb` numeric |
| `anchor_direction_followed_moved`   | `1` iff `(pa-pb)·(z-pb) > 0` AND `pa != pb` AND all numeric |
| `pred_b_equal_anchor`               | `1` iff `pb == z` |
| `numeric_distance_to_anchor`        | `|pa - z|` when both are int-parseable; `NaN` otherwise |

We compute each metric on **two subsets** of samples:

- **broad** — all samples in the cell (`base-wrong ∪ base-correct`).
- **base-wrong** — only samples where `b` arm `exact_match == 0`, i.e., the
  cohort whose baseline answer is already incorrect. This is the cohort
  where anchoring has signal: a model that already had `gt` at baseline
  almost never moves toward the anchor.

The canonical aggregator `scripts/build_e5e_e7_5dataset_summary.py` writes
**base-wrong** to `main_panel_5dataset_per_cell.csv` — that's the cohort
the paper §4 / Appendix D tables refer to. Below we show both so a reviewer
can see how much the cohort choice matters.


In [11]:
ARMS = {
    "a": "target_plus_irrelevant_number_S1",
    "m": "target_plus_irrelevant_number_masked_S1",
}


def _aggregate(arm_rows: pd.DataFrame) -> dict:
    # Aggregate canonical per-row flags into adopt / df / em rates.
    n            = len(arm_rows)
    n_pb_ne_anc  = int((arm_rows["pred_b_equal_anchor"] == 0).sum())
    n_num_anchor = int(arm_rows["numeric_distance_to_anchor"].notna().sum())
    adopt_num    = int(arm_rows["anchor_adopted"].sum())
    df_num       = int(arm_rows["anchor_direction_followed_moved"].sum())
    em_num       = int(arm_rows["exact_match"].sum())
    return {
        "n":     n,
        "adopt": adopt_num / n_pb_ne_anc  if n_pb_ne_anc  else np.nan,
        "df":    df_num    / n_num_anchor if n_num_anchor else np.nan,
        "em":    em_num    / n            if n            else np.nan,
    }


def per_cell_metrics(sub: pd.DataFrame) -> dict:
    # Identify base-wrong sids from the target_only rows.
    base = sub[sub["condition"] == "target_only"]
    base_wrong_sids = set(base.loc[base["exact_match"] == 0, "sample_instance_id"])

    out = {"n_base_wrong": len(base_wrong_sids)}
    for arm_code, arm_cond in ARMS.items():
        arm_all = sub[sub["condition"] == arm_cond]
        arm_wb  = arm_all[arm_all["sample_instance_id"].isin(base_wrong_sids)]

        for tag, rows in (("broad", arm_all), ("wb", arm_wb)):
            agg = _aggregate(rows)
            for metric, value in agg.items():
                out[f"{metric}_{arm_code}_{tag}"] = value
    return out


cell_rows = []
for ds in DATASETS:
    for model in MODELS:
        sub = ALL[(ALL["dataset"] == DISPLAY[ds]) & (ALL["model"] == model)]
        m = per_cell_metrics(sub)
        cell_rows.append({"dataset": DISPLAY[ds], "model": model, **m})

PER_CELL = pd.DataFrame(cell_rows)
PER_CELL.head(6)


,dataset,model,n_base_wrong,n_a_broad,adopt_a_broad,df_a_broad,em_a_broad,n_a_wb,adopt_a_wb,df_a_wb,em_a_wb,n_m_broad,adopt_m_broad,df_m_broad,em_m_broad,n_m_wb,adopt_m_wb,df_m_wb,em_m_wb
0,TallyQA,llava-onevision-qwen2-7b-ov,8178,38245,0.011918,0.039367,0.776258,8178,0.032027,0.099057,0.117021,38245,0.008833,0.035236,0.776572,8178,0.022054,0.087457,0.102959
1,TallyQA,llava-next-interleaved-7b,11209,38245,0.026433,0.065627,0.698209,11209,0.050348,0.127289,0.124989,38245,0.014470,0.056396,0.700876,11209,0.028551,0.109344,0.117495
2,TallyQA,qwen2.5-vl-7b-instruct,7541,38245,0.010512,0.029023,0.791188,7541,0.030041,0.084869,0.109800,38245,0.010694,0.029886,0.793123,7541,0.032620,0.089113,0.116430
3,TallyQA,qwen2.5-vl-32b-instruct,7407,38245,0.012795,0.035746,0.794927,7407,0.037924,0.109251,0.140543,38245,0.011829,0.034129,0.795581,7407,0.034853,0.105121,0.142028
4,TallyQA,gemma3-4b-it,14772,38245,0.032975,0.098235,0.629520,14772,0.061894,0.172285,0.173978,38245,0.025866,0.086913,0.630514,14772,0.048592,0.156038,0.162063
5,TallyQA,gemma3-27b-it,11014,38245,0.026658,0.072637,0.706916,11014,0.058683,0.151625,0.139550,38245,0.015578,0.059720,0.716460,11014,0.033696,0.132559,0.148629


## 12 · Paper Appendix D.1 — direction-follow (a-arm) %

Each cell shows **`broad / base-wrong`** — `broad` aggregates over all
samples in the cell, `base-wrong` restricts to the subset whose `b`-arm
prediction differs from GT. The right-hand value matches the canonical
`main_panel_5dataset_per_cell.csv` and the numbers reported in outline
§D.1 (per-model mean range on the base-wrong cohort: **14.6 – 33.0**).
The left-hand value is the *fraction of all responses* that direction-follow,
diluted by base-correct samples whose answers rarely move toward an anchor.


In [12]:
MODEL_DISPLAY = {
    "llava-onevision-qwen2-7b-ov":  "OneVision-7B",
    "llava-next-interleaved-7b":    "Interleave-7B",
    "qwen2.5-vl-7b-instruct":       "Qwen2.5-VL-7B",
    "qwen2.5-vl-32b-instruct":      "Qwen2.5-VL-32B",
    "gemma3-4b-it":                 "Gemma3-4B",
    "gemma3-27b-it":                "Gemma3-27B",
}
DATASET_DISPLAY_ORDER = ["TallyQA", "ChartQA", "MathVista", "PlotQA", "InfographicVQA"]


def _pivot_pct(metric_col: str) -> pd.DataFrame:
    t = PER_CELL.pivot(index="model", columns="dataset", values=metric_col) * 100
    t = t.reindex(index=MODELS, columns=DATASET_DISPLAY_ORDER)
    t.index = [MODEL_DISPLAY[m] for m in t.index]
    t["Mean"] = t.mean(axis=1)
    return t.round(1)


def _pivot_pct_paired(metric: str, arm: str = "a") -> pd.DataFrame:
    # broad / base-wrong combined, e.g. "1.2 / 9.9", per (model, dataset) cell.
    broad = _pivot_pct(f"{metric}_{arm}_broad")
    wb    = _pivot_pct(f"{metric}_{arm}_wb")
    return broad.astype(str).combine(wb.astype(str), lambda a, b: a + " / " + b)


D1_paired = _pivot_pct_paired("df", "a")
D1_wb     = _pivot_pct("df_a_wb")
D1_broad  = _pivot_pct("df_a_broad")
print("D.1 · df(a) % — each cell is  broad / base-wrong")
print(f"  per-model mean range, broad:      {D1_broad['Mean'].min():.1f} – {D1_broad['Mean'].max():.1f}")
print(f"  per-model mean range, base-wrong: {D1_wb['Mean'].min():.1f} – {D1_wb['Mean'].max():.1f}   (outline expects 14.6 – 33.0)")
D1_paired


D.1 · df(a) % — each cell is  broad / base-wrong
  per-model mean range, broad:      4.4 – 23.1
  per-model mean range, base-wrong: 14.6 – 33.0   (outline expects 14.6 – 33.0)


dataset,TallyQA,ChartQA,MathVista,PlotQA,InfographicVQA,Mean
OneVision-7B,3.9 / 9.9,6.6 / 18.9,15.3 / 20.5,10.9 / 20.6,11.2 / 19.0,9.6 / 17.8
Interleave-7B,6.6 / 12.7,15.2 / 20.0,20.5 / 26.6,27.2 / 29.5,22.3 / 24.4,18.4 / 22.6
Qwen2.5-VL-7B,2.9 / 8.5,5.1 / 18.7,7.2 / 16.2,4.0 / 17.4,3.1 / 12.3,4.4 / 14.6
Qwen2.5-VL-32B,3.6 / 10.9,5.0 / 20.3,8.3 / 18.4,4.0 / 16.3,3.8 / 15.6,4.9 / 16.3
Gemma3-4B,9.8 / 17.2,19.0 / 34.6,34.5 / 41.3,28.7 / 39.5,23.4 / 32.4,23.1 / 33.0
Gemma3-27B,7.3 / 15.2,9.6 / 24.0,21.6 / 33.2,11.7 / 22.7,18.3 / 35.0,13.7 / 26.0


## 13 · Paper Appendix D.2 — adopt (a-arm) %

Same `broad / base-wrong` convention as §12. Outline §D.2 (base-wrong
cohort) headline: per-model mean range **2.8 – 15.4**.


In [13]:
D2_paired = _pivot_pct_paired("adopt", "a")
D2_wb     = _pivot_pct("adopt_a_wb")
D2_broad  = _pivot_pct("adopt_a_broad")
print("D.2 · adopt(a) % — each cell is  broad / base-wrong")
print(f"  per-model mean range, broad:      {D2_broad['Mean'].min():.1f} – {D2_broad['Mean'].max():.1f}")
print(f"  per-model mean range, base-wrong: {D2_wb['Mean'].min():.1f} – {D2_wb['Mean'].max():.1f}   (outline expects 2.8 – 15.4)")
D2_paired


D.2 · adopt(a) % — each cell is  broad / base-wrong
  per-model mean range, broad:      1.4 – 12.5
  per-model mean range, base-wrong: 2.8 – 15.3   (outline expects 2.8 – 15.4)


dataset,TallyQA,ChartQA,MathVista,PlotQA,InfographicVQA,Mean
OneVision-7B,1.2 / 3.2,1.5 / 3.3,10.8 / 8.4,5.9 / 9.0,2.3 / 2.0,4.3 / 5.2
Interleave-7B,2.6 / 5.0,2.8 / 3.2,6.6 / 6.5,8.1 / 8.2,6.0 / 6.1,5.2 / 5.8
Qwen2.5-VL-7B,1.1 / 3.0,1.7 / 3.5,2.0 / 2.6,1.3 / 2.4,0.9 / 2.5,1.4 / 2.8
Qwen2.5-VL-32B,1.3 / 3.8,1.7 / 4.3,8.0 / 12.8,1.1 / 2.3,2.8 / 9.8,3.0 / 6.6
Gemma3-4B,3.3 / 6.2,5.9 / 8.8,27.6 / 30.1,15.4 / 18.4,10.5 / 13.3,12.5 / 15.3
Gemma3-27B,2.7 / 5.9,3.7 / 7.0,17.6 / 23.0,6.3 / 9.9,10.3 / 16.3,8.1 / 12.4


## 14 · Cross-check against canonical aggregator CSV

`docs/insights/_data/main_panel_5dataset_per_cell.csv` is produced by
`scripts/build_e5e_e7_5dataset_summary.py`. We should match it cell-by-cell
within numerical tolerance (≤ 0.05 pp).


In [14]:
canon = pd.read_csv(CANON_PER_CELL)
canon_a = canon[canon["cond_class"] == "a"].copy()
canon_a["adopt_pct"] = canon_a["adopt_M2"] * 100
canon_a["df_pct"]    = canon_a["direction_follow_M2"] * 100
canon_a["em_pct"]    = canon_a["exact_match"] * 100

ours = PER_CELL.copy()
# Canonical CSV is computed on the base-wrong cohort — compare against the
# `_wb` columns. The `_broad` columns above are reported alongside for the
# reviewer; they have no canonical counterpart.
ours["adopt_pct"] = ours["adopt_a_wb"] * 100
ours["df_pct"]    = ours["df_a_wb"]    * 100
ours["em_pct"]    = ours["em_a_wb"]    * 100

merged = canon_a.merge(
    ours[["dataset", "model", "adopt_pct", "df_pct", "em_pct"]],
    on=["dataset", "model"],
    suffixes=("_canon", "_ours"),
)
for metric in ("adopt_pct", "df_pct", "em_pct"):
    diff = (merged[f"{metric}_canon"] - merged[f"{metric}_ours"]).abs()
    max_diff = diff.max()
    print(f"  {metric:<10}  max |Δ| = {max_diff:.4f} pp   "
          f"({'OK' if max_diff < 0.05 else 'INVESTIGATE'})")

merged[["dataset", "model",
        "adopt_pct_canon", "adopt_pct_ours",
        "df_pct_canon", "df_pct_ours",
        "em_pct_canon", "em_pct_ours"]].head(10)


  adopt_pct   max |Δ| = 0.0000 pp   (OK)
  df_pct      max |Δ| = 0.0000 pp   (OK)
  em_pct      max |Δ| = 0.0000 pp   (OK)


,dataset,model,adopt_pct_canon,adopt_pct_ours,df_pct_canon,df_pct_ours,em_pct_canon,em_pct_ours
0,TallyQA,gemma3-27b-it,5.868326,5.868326,15.162520,15.162520,13.954966,13.954966
1,TallyQA,gemma3-4b-it,6.189358,6.189358,17.228540,17.228540,17.397780,17.397780
2,TallyQA,llava-next-interleaved-7b,5.034793,5.034793,12.728897,12.728897,12.498885,12.498885
3,TallyQA,llava-onevision-qwen2-7b-ov,3.202697,3.202697,9.905718,9.905718,11.702128,11.702128
4,TallyQA,qwen2.5-vl-32b-instruct,3.792415,3.792415,10.925051,10.925051,14.054273,14.054273
5,TallyQA,qwen2.5-vl-7b-instruct,3.004096,3.004096,8.486938,8.486938,10.979976,10.979976
6,ChartQA,gemma3-27b-it,7.042254,7.042254,24.000000,24.000000,11.353712,11.353712
7,ChartQA,gemma3-4b-it,8.771930,8.771930,34.615385,34.615385,10.101010,10.101010
8,ChartQA,llava-next-interleaved-7b,3.233831,3.233831,20.000000,20.000000,4.086538,4.086538
9,ChartQA,llava-onevision-qwen2-7b-ov,3.317536,3.317536,18.905473,18.905473,7.079646,7.079646


## Summary

Notebook ran top-to-bottom: spec-compliance ✓, anchor distance ✓, per-cell metrics ✓,
canonical match ≤ 0.05 pp. This is the evidence chain backing paper §4 and Appendix D.

**To re-run from scratch (model inference):** use `configs/experiment_e5e_*_full.yaml`
(TallyQA / ChartQA / MathVista) and `configs/experiment_e7_*_full.yaml`
(PlotQA / InfographicVQA), then re-aggregate via
`scripts/build_e5e_e7_5dataset_summary.py`.
